# 🤖 Agents — the brains, and the fence around them

Every organ you have met so far is deterministic: the contract's checks replay
identically (`06`/`07`), the bouncer's checklist is pure arithmetic (`08`), the hands
apply exactly the config they are told (`09`). One thing is still missing from the
story: **who decides to buy?** Somebody has to look at Bell's offer — 50 Mbps for
10 TOK — and judge whether it is worth it. That judgment is this chapter, and it is
deliberately caged: **hard rule 1** says LLM judgment exists in exactly two places
(the consumer's accept/reject, the provider's quote/decline) and nowhere else. By the
end you will *prove* that count with a scan, not take it on faith.

The whole chapter runs on **stub LLMs and stub tools** — the exact stand-ins the
repo's own tests use — so you need no model server, no chain, no router. The graph
structure you poke is byte-for-byte what the live agents run; only the two judgment
slots swap a script for a real model (the closing recipe shows how).

**What you'll be able to do after this notebook**

- Explain what an LLM honestly is, and read a chat-completions request/response by shape
- Run the validate-and-retry loop that turns model *text* into validated pydantic *types* — and exhaust it on purpose
- Build a LangGraph from zero, then drive both branches of the consumer graph and the provider's no-overselling gate
- Say where MCP and A2A fit: tools that keep keys in `chainmcp` (rule 2), and discovery cards over a trustless wire
- Point at the exactly-two cells where judgment lives, and say why a fooled LLM still can't misconfigure the network

**You need:** notebooks 00–09 (especially 02's pydantic fence, 03's Protocols, 05's
canned judgment slots, 07's signatures, 08's challenge–response). No extra
infrastructure — this chapter is plain in-process Python.

**Estimated time:** 75–100 minutes.

> The compact one-pass version of this tour lives at
> [`e2e/notebooks/agents_explore.ipynb`](../agents_explore.ipynb) — worth a replay
> *after* you finish here.

## 1. What an LLM is, and the shape of a conversation

One honest paragraph, no mystique: a **large language model (LLM)** is a next-word
predictor. It was trained on enormous amounts of text to answer one question, over and
over: *given these words so far, what word plausibly comes next?* Generate the next
word, append it, repeat — that is the whole trick. It turns out that a good enough
next-word predictor, shown an offer and asked "accept or reject, with a reason?",
continues with text that *reads like a sensible decision*. It is good at
**judgment-shaped text**. It does not *know* the window is 14:00–16:00 the way the
predicate knows it (08) — which is exactly why this chapter builds a fence around it.

How you talk to one: the **chat-completions API** — an HTTP endpoint (you met
endpoints in 08) that takes a JSON list of **messages** and returns the model's
continuation. Each message has a **role**: `system` (standing instructions — who the
model is supposed to be), `user` (the question), and `assistant` (what the model said
back). A conversation is literally a list of dicts:

In [ ]:
messages = [
    {"role": "system", "content": "You are a procurement agent buying network services."},
    {"role": "user", "content": "OFFER: 50 Mbps for 10 TOK. BUDGET: 15 TOK. Accept?"},
]
for m in messages:
    print(f"{m['role']:>7} → {m['content']}")
assert [m["role"] for m in messages] == ["system", "user"]
print("\n✓ a conversation is a list of role+content dicts — that is the entire request shape")

And what comes back — the **response object**, again just a dict shape. Two things to
look for: the reply text hides at `choices[0].message.content` (`choices` is a list
because you *may* ask for several alternative continuations; this repo always takes
the first), and `usage` counts **tokens** — the model's unit of text, a word piece of
roughly four characters. Models read and write tokens, and hosted backends bill by
them.

In [ ]:
response = {
    "id": "chatcmpl-42",
    "model": "qwen3:4b",
    "choices": [
        {"index": 0,
         "message": {"role": "assistant",
                     "content": '{"accept": true, "reason": "10 TOK is within the 15 TOK budget"}'}}
    ],
    "usage": {"prompt_tokens": 58, "completion_tokens": 19},
}
reply = response["choices"][0]["message"]["content"]
print("assistant said →", reply)
print("cost           →", response["usage"], " ← tokens: the LLM's billing unit")
assert response["choices"][0]["message"]["role"] == "assistant"
print("✓ request = messages list, response = choices[0].message.content — the whole API")

**✏️ Your turn 1.1 — build the provider's side of the conversation**

Section 7 will show Bell's pricing slot; build its raw API shape now. Add the missing
`user` message asking the model to price *60 Mbps from hostA to hostB*, then pull the
assistant's reply out of the prepared response dict. Success: the cell prints
`provider replied → 12 TOK` and both TODOs are gone.

In [ ]:
pricing_response = {
    "choices": [{"index": 0, "message": {"role": "assistant", "content": "12 TOK"}}],
    "usage": {"prompt_tokens": 41, "completion_tokens": 4},
}
pricing_messages = [
    {"role": "system", "content": "You are a network provider. Quote a fair price in whole TOK."},
    # TODO: append the *user* message asking to price 60 Mbps, hostA -> hostB
]
reply = "?"  # TODO: pull the assistant's reply out of pricing_response
print("provider replied →", reply)
# assert reply == "12 TOK" and len(pricing_messages) == 2   # un-comment when done

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
pricing_messages.append(
    {"role": "user", "content": "Price this request: 60 Mbps, hostA -> hostB."}
)
reply = pricing_response["choices"][0]["message"]["content"]
print("provider replied →", reply)
assert reply == "12 TOK" and len(pricing_messages) == 2
```

The user message carries the question and the reply always lives at
`choices[0].message.content` — every cell in this chapter builds on exactly this shape.
</details>

## 2. One client, any backend (ADR-001)

What breaks without this section: imagine `agents` imported an SDK specific to
**Ollama** — a free program that runs small language models on your own laptop. The
day the model moves to a rented GPU, every agent file changes. So **ADR-001** rules:
agent code speaks only the *OpenAI-compatible* chat dialect (the request/response
shape you just built — Ollama, **vLLM** (a fast open-source model server) and most
other servers all serve it), and the backend is chosen by **environment variables,
never by code**. One more name before the table: **Modal** is a serverless GPU cloud —
the repo's `llmserve/` package deploys the agents' vLLM there, and a GPU boots on
demand when a request arrives:

| env var | local Ollama | Modal-hosted vLLM (the live-demo leg) |
|---|---|---|
| `LLM_BASE_URL` | `http://localhost:11434/v1` | `https://<workspace>--a2a-llm-serve.modal.run/v1` |
| `LLM_MODEL` | `qwen3:4b` | `qwen3-4b` |
| `LLM_API_KEY` | `ollama` (ignored) | the Modal proxy token |

(That Modal model name is the *served alias*, not the weights' full name:
`llmserve/modal_llm.py` starts vLLM with `--served-model-name qwen3-4b` — its
`SERVED_AS` constant — so `qwen3-4b` is what clients put in `LLM_MODEL`. ADR-001's
original table, written before that deployment existed, still shows the older name.)

`agents/src/agents/llm.py` is the **only** file in `agents` that imports an LLM SDK —
and even that one is the generic `openai` client used purely as a speaker of the
dialect, not a lock-in to OpenAI-the-company. The config is a frozen dataclass (01)
with a `@classmethod` alternate constructor that reads the three vars:

In [ ]:
import dataclasses
import inspect

from agents.llm import LLMConfig

print(inspect.getsource(LLMConfig))
cfg = LLMConfig.from_env()
print("this machine resolves →", cfg.base_url, "|", cfg.model)
assert dataclasses.is_dataclass(cfg) and cfg.max_retries == 3 and cfg.temperature == 0.0

Note `temperature: float = 0.0` — **temperature** is how much randomness goes into
picking each next token; `0.0` means "always the most likely one". Decisions want
determinism, not creativity.

Now the client itself. One fact worth pinning because every notebook and test relies
on it: **constructing `LLMClient` never contacts a server**. The underlying `OpenAI(...)`
object is lazy — nothing dials until you actually ask for a completion. Build one
against a hostname that cannot exist (names with a leading `_` are internal — we peek
only to learn):

In [ ]:
from agents.llm import LLMClient

bogus = LLMClient(LLMConfig(base_url="http://nowhere.invalid/v1", model="ghost", api_key="k"))
print("client built against →", bogus._config.base_url, " ← no connection was attempted")
print("breadcrumbs          →", bogus.last_attempts, "attempts,", bogus.last_usage or "{} usage")
assert bogus.last_attempts == 0 and bogus.last_usage == {}
print("✓ construction is free; only a real call would fail — that is why stubs can drop in")

**✏️ Your turn 2.1 — break the frozen config**

Build the Modal-leg config by hand, then try to mutate `modal_cfg.model` to something
else and catch what fires. Success: you can name the exception class — the error name
tells you which guarantee (from 01's frozen dataclasses) protected the config.

In [ ]:
modal_cfg = LLMConfig(
    base_url="https://myapp--llm.modal.run/v1",   # the Modal leg from ADR-001's table
    model="Qwen/Qwen3-4B",
    api_key="proxy-token",
)
print(modal_cfg.model, "@", modal_cfg.base_url)
caught = None
# TODO: inside a try/except, assign modal_cfg.model = "gpt-4" and store the exception in `caught`
# assert isinstance(caught, dataclasses.FrozenInstanceError)   # un-comment when done

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
try:
    modal_cfg.model = "gpt-4"
except dataclasses.FrozenInstanceError as e:
    caught = e
    print("✗ FrozenInstanceError:", e)
assert isinstance(caught, dataclasses.FrozenInstanceError)
```

`@dataclass(frozen=True)` makes the config immutable, so a backend choice can never be
silently edited mid-run — swap backends by *constructing* a new config (or new env vars).
</details>

## 3. The fence: text in, types out

Here is the structured-output problem in one sentence: the consumer graph must branch
on `decision.accept` — a `bool`, not a vibe — but **every LLM returns text**. Small
local models make it worse: they wrap JSON in prose, in ` ```json ` fences, in
`<think>…</think>` blocks, or emit JSON that *parses* but is missing a required field.
Backends offer "JSON modes" that all behave differently — exactly the lock-in ADR-001
forbids.

The repo's answer is `LLMClient.structured(system, user, schema)`: ask for JSON
matching a **pydantic schema**, then **validate the reply in code and retry** until it
parses or the budget (3 attempts) runs out. The caller gets a guaranteed-valid object
or a clean `StructuredError` — never a raw string. Pydantic, which you learned as the
border guard for payloads in 02, is here the fence around the model.

First: what the model is actually shown. `structured()` appends the target schema —
`DecisionOutput.model_json_schema()` — to the system prompt. One deliberate edit: it
pops the top-level `description` (the class docstring), because literal-minded small
models were observed echoing the schema *document* back instead of producing an
instance.

In [ ]:
import json

from a2a_interfaces import DecisionOutput

schema_json = DecisionOutput.model_json_schema()
print("description present before the pop?", "description" in schema_json)
schema_json.pop("description", None)          # the same line structured() runs
print(json.dumps(schema_json, indent=2))
assert "description" not in schema_json and schema_json["required"] == ["accept", "reason"]

Now read the loop itself — the load-bearing ten lines of the whole chapter. Look for:
the schema instruction glued onto the system prompt, `attempts.append(reply)` keeping
evidence, `model_validate_json(_extract_json(reply))` as the fence, and
`except (ValidationError, ValueError): continue` quietly burning one retry per bad
reply.

In [ ]:
print(inspect.getsource(LLMClient.structured))

Before running that loop with no model attached, one new language construct — the
stand-ins below lean on it. So far you have only met `type(x)` with **one** argument
("what is this thing?"). With **three** arguments — `type(name, parent_classes,
{attribute: value})` — it *builds a class on the spot*, no `class` statement needed.
Perfect for faking an object that only has to *have* one attribute:

In [ ]:
Point = type("Point", (), {"x": 1})          # same as:  class Point: x = 1
print("Point().x       →", Point().x)
Reply = type("R", (), {"content": "hi"})     # a fake reply that ONLY has .content
print("Reply().content →", Reply().content)
assert Point().x == 1 and Reply().content == "hi"

Run it — without any model. The repo's own `agents/tests/test_llm_retry.py` stands in
a **scripted fake OpenAI client**: `_ScriptedChat` replays a fixed list of "assistant
replies", and `_client_with` splices it into a *real* `LLMClient` in place of the
network. We reuse those helpers name-for-name (the nested three-argument `type(...)` classes
you just met build a minimal response object — exactly the `choices[0].message.content`
shape from §1, and deliberately with **no** `usage` attribute).

In [ ]:
from agents.llm import StructuredError, _extract_json


class _ScriptedChat:
    """Replays a fixed list of assistant replies, one per call."""

    def __init__(self, replies):
        self._replies = list(replies)
        self.calls = 0

    def create(self, **_kwargs):
        reply = self._replies[self.calls]
        self.calls += 1
        return type(
            "R", (), {"choices": [type("C", (), {"message": type("M", (), {"content": reply})})]}
        )


def _client_with(replies):
    client = LLMClient(LLMConfig(base_url="x", model="m", api_key="k", max_retries=3))
    chat = _ScriptedChat(replies)
    client._client = type("O", (), {"chat": type("Ch", (), {"completions": chat})})()
    return client, chat


print("scripted backend ready — same names as agents/tests/test_llm_retry.py")

The script: attempt 1 is not JSON at all; attempt 2 **is valid JSON but missing
`reason`** — watch closely, this one is refused by the *schema*, not the JSON parser;
attempt 3 validates. First we judge each reply by hand, then let `structured()` run
the real loop and land on attempt 3:

In [ ]:
from pydantic import ValidationError

replies = ["not json at all", '{"accept": false}', '{"accept": false, "reason": "too pricey"}']
for i, reply in enumerate(replies, 1):
    try:
        DecisionOutput.model_validate_json(_extract_json(reply))
        verdict = "validates ✓"
    except ValidationError as e:
        err = e.errors()[0]
        verdict = f"✗ {err['type']}  {err['loc'] or ''}"
    print(f"attempt {i}: {reply!r:<50} → {verdict}")

client, chat = _client_with(replies)
out = client.structured("sys", "usr", DecisionOutput)
print(f"\nstructured() → {out!r}   after {client.last_attempts} attempts")
assert out.reason == "too pricey" and chat.calls == 3 and client.last_attempts == 3
assert client.last_usage == {}   # scripted replies carry no .usage — llm.py's getattr guard

Before validating, each reply passes through `_extract_json` — the wrapper-stripper
(a `_private` helper; peeking to learn). It drops everything before a closing
`</think>`, then walks the text with a **brace-depth counter** to cut out the first
*balanced* `{...}` — not merely "up to the first `}`", which would truncate nested
objects. Throw all three wrappers at it at once:

In [ ]:
wrapped = '<think>hmm, budgets…</think>here you go: ```json\n{"a": {"b": 2}}\n``` hope that helps'
print("survives all three wrappers →", repr(_extract_json(wrapped)))
assert _extract_json(wrapped) == '{"a": {"b": 2}}'

for case in ['{"a": 1}', '```json\n{"a": 1}\n```', 'sure: {"a": 1} enjoy', '<think>…</think>\n{"a": 1}']:
    assert _extract_json(case) == '{"a": 1}'
print('✓ prose, fences, <think> blocks: stripped — and nested {} kept whole by the depth counter')

Break it on purpose, twice. First, exhaust the budget: three junk replies leave
`structured()` no choice but a `StructuredError` — which *carries the attempts*, so a
caller (or you) can read what the model actually said. Second, feed the extractor an
unbalanced blob: it returns the tail unchanged — **the extractor tolerates, the
validator decides**. There is exactly one gatekeeper, and it is pydantic.

In [ ]:
client, chat = _client_with(["nope", "still nope", "nope again"])
try:
    client.structured("sys", "usr", DecisionOutput)
    raise AssertionError("should have raised")
except StructuredError as err:
    print("✗ StructuredError:", err)
    print("   evidence — what the 'model' said:", err.attempts)
    assert len(err.attempts) == 3 and chat.calls == 3

truncated = '{"a": {"b": 2}'                       # closing brace lost in transit
print("\nextractor on an unbalanced blob →", repr(_extract_json(truncated)), " ← unchanged")
try:
    DecisionOutput.model_validate_json(_extract_json(truncated))
except ValidationError as e:
    print("✗ the VALIDATOR says no:", e.errors()[0]["type"])
assert _extract_json(truncated) == truncated

**✏️ Your turn 3.1 — make it succeed on attempt 2, exactly**

Sabotage the first reply so it *fails validation* (your choice: not JSON at all, or
valid JSON missing a required field), keep the second valid, and predict
`client.last_attempts` before running. Success: the print says `succeeded on attempt 2`
and the un-commented assert passes.

In [ ]:
# my prediction: last_attempts will be ___
replies = [
    '{"accept": true, "reason": "first try"}',    # TODO: sabotage this one so validation fails
    '{"accept": false, "reason": "second try"}',  # this one should win
    "unused filler",
]
client, chat = _client_with(replies)
out = client.structured("sys", "usr", DecisionOutput)
print(f"succeeded on attempt {client.last_attempts} → {out!r}")
# assert client.last_attempts == 2 and chat.calls == 2   # un-comment when done

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
replies = [
    '{"accept": true}',                           # valid JSON, but `reason` is missing
    '{"accept": false, "reason": "second try"}',
    "unused filler",
]
client, chat = _client_with(replies)
out = client.structured("sys", "usr", DecisionOutput)
print(f"succeeded on attempt {client.last_attempts} → {out!r}")
assert client.last_attempts == 2 and chat.calls == 2
```

Both sabotage flavors land in the same `except (ValidationError, ValueError)` arm —
the loop doesn't care *why* a reply failed, only that the fence refused it.
</details>

## 4. `decision.py` — the consumer's one judgment slot

Rule 1, restated precisely: **judgment proposes, the predicate disposes.** The LLM you
are about to meet can only *propose* a purchase. Money still moves only if the
contract's checks pass (06/07), and the network changes only if the bouncer's
deterministic checklist says yes (08). A fooled, hallucinating, or malicious model has
a bounded blast radius: it can buy something unwise; it can never misconfigure a
router.

`decide()` is one of the exactly-two slots, and it is small enough to read whole.
Watch for three things: the wei→TOK floor division (`int(price) // 10**18` — 04's
integer money resurfacing), the prompt built from `model_dump_json()` of the *real*
payloads, and the `except StructuredError` fallback that turns a fence failure into a
**safe decline**:

In [ ]:
import agents.decision as decision_mod

print(inspect.getsource(decision_mod))

See the exact prompt a real model would receive. Any object with a
`structured(system, user, schema)` method fits where `LLMClient` goes — the same duck
typing the tests lean on (03's structural fit, without even a Protocol to point to).
This spy records the prompt and answers with a script:

In [ ]:
from a2a_interfaces.fixtures import BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER

from agents.decision import decide


class _SpyLLM:
    """Duck-typed LLMClient stand-in: records the exact prompt, answers with a script."""

    def structured(self, system, user, schema):
        self.system, self.user, self.schema = system, user, schema
        return DecisionOutput(accept=True, reason="scripted")


spy = _SpyLLM()
verdict = decide(spy, BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER, budget_tok=15)
print("SYSTEM →", spy.system[:78] + "…")
print("\nUSER — the exact prompt a real model would see:\n")
print(spy.user)
assert spy.schema is DecisionOutput and "Price: 10 TOK" in spy.user and verdict.accept

The whole canonical example is in there: the NEED (Ada's 50 Mbps, hostA→hostB, the
14:00–16:00 window) and the OFFER (Bell's address, the params blob you dissected by
hand in 04, the 10-TOK price as a wei string) — plus the one line of judgment
material: `Price: 10 TOK.  Budget: 15 TOK.`

Now break the slot and watch the safe default. A fake whose `structured()` always
raises `StructuredError` simulates a model that never produced valid JSON. The
invariant worth memorizing: **a schema failure may decline, but can never accept** —
a procurement agent that cannot read an offer must not buy it.

In [ ]:
class _BrokenLLM:
    def structured(self, system, user, schema):
        raise StructuredError(["gibberish", "gibberish", "gibberish"])


fallback = decide(_BrokenLLM(), BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER, budget_tok=15)
print("verdict →", fallback.model_dump())
assert fallback.accept is False
print("✓ fence failure → safe DECLINE, never a crash, never an accept")

**✏️ Your turn 4.1 — starve the budget, then think about who answered**

Write your prediction as a comment first: with `budget_tok=5` against the 10-TOK
offer, does `decide()` return `accept=True` or `accept=False`? Then change the budget
and check the `Budget:` line in the captured prompt. Success: the prompt says
`Budget: 5 TOK` — and the verdict tells you something important about *who* is
answering.

In [ ]:
# my prediction: with budget_tok=5, decide() returns accept=___   (think: WHO answers?)
spy5 = _SpyLLM()
verdict5 = decide(spy5, BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER, budget_tok=15)  # TODO: 15 → 5
print([line for line in spy5.user.splitlines() if "Budget" in line][0])
print("verdict →", verdict5.model_dump())
# assert "Budget: 5 TOK" in spy5.user and verdict5.accept   # un-comment when done

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
spy5 = _SpyLLM()
verdict5 = decide(spy5, BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER, budget_tok=5)
print([line for line in spy5.user.splitlines() if "Budget" in line][0])
print("verdict →", verdict5.model_dump())
assert "Budget: 5 TOK" in spy5.user and verdict5.accept
```

Still `accept=True` — the spy never reads the prompt. The budget line reaches the
model verbatim, but only a *real* model weighs it (the closing recipe arms that; the
tests stub it for exactly this determinism).
</details>

## 5. LangGraph from zero

The buy flow is a *sequence with a branch*: get a quote, decide, then either the
purchase path or a graceful exit. You met "state machine as data" in 08 (the
controller's session states); **LangGraph** is a library for wiring exactly that shape
out of plain Python:

- **state** — one object that flows through the run; each step reads it and fills in a bit more
- **node** — a plain function `state → state` (no magic; you could call it yourself)
- **edge** — "after this node, run that one"; `END` is the built-in exit sentinel
- **conditional edge** — a router function picks the next node by looking at the state
- **compile / invoke** — `.compile()` freezes the wiring into a runnable; `.invoke(state)` runs it to `END`

Toy first (the construct ladder): two nodes, one straight line.

In [ ]:
from dataclasses import dataclass, field

from langgraph.graph import END, StateGraph


@dataclass
class ToyState:
    name: str
    greeting: str | None = None


def make_greeting(state: ToyState) -> ToyState:   # a node: plain function, state in, state out
    state.greeting = f"hello, {state.name}"
    return state


def shout(state: ToyState) -> ToyState:
    state.greeting = state.greeting.upper()
    return state


toy = StateGraph(ToyState)
toy.add_node("make_greeting", make_greeting)
toy.add_node("shout", shout)
toy.set_entry_point("make_greeting")
toy.add_edge("make_greeting", "shout")
toy.add_edge("shout", END)
result = toy.compile().invoke(ToyState(name="Ada"))
print(result)

One gotcha to pin *now*, because it bites everyone once: you hand `invoke()` your
dataclass, but it hands back a **plain dict**. `result["greeting"]` works;
`result.greeting` is an `AttributeError`:

In [ ]:
print("type(result) →", type(result).__name__)
try:
    result.greeting
    raise AssertionError("should have failed")
except AttributeError as e:
    print("✗ result.greeting →", e)
print("result['greeting'] →", result["greeting"], " ← dict access, always")
assert isinstance(result, dict) and result["greeting"] == "HELLO, ADA"

Now the branch — the construct the consumer graph's accept/decline runs on.
`add_conditional_edges(source, router, mapping)` takes a router function returning a
*label* and a mapping from labels to node names:

In [ ]:
@dataclass
class GateState:
    n: int
    verdict: str | None = None


def check(state):
    return state                                   # a node may just pass the state along


def small(state):
    state.verdict = "small"
    return state


def big(state):
    state.verdict = "big"
    return state


gate = StateGraph(GateState)
for name, fn in [("check", check), ("small", small), ("big", big)]:
    gate.add_node(name, fn)
gate.set_entry_point("check")
gate.add_conditional_edges("check", lambda s: "small" if s.n < 10 else "big",
                           {"small": "small", "big": "big"})
gate.add_edge("small", END)
gate.add_edge("big", END)
gate_graph = gate.compile()
print("n=3  →", gate_graph.invoke(GateState(n=3))["verdict"])
print("n=50 →", gate_graph.invoke(GateState(n=50))["verdict"])
assert gate_graph.invoke(GateState(n=3))["verdict"] == "small"

**✏️ Your turn 5.1 — add a third branch**

Extend the gate so `n == 10` lands in a new `"exact"` node. Four TODOs are marked:
add the node, teach the router, extend the mapping, wire the exit edge. Success:
`n=10 → exact` (right now it prints `big`).

In [ ]:
def exact(state):
    state.verdict = "exact"
    return state


g3 = StateGraph(GateState)
for name, fn in [("check", check), ("small", small), ("big", big)]:
    g3.add_node(name, fn)
# TODO 1: g3.add_node("exact", exact)
g3.set_entry_point("check")


def router(s):
    # TODO 2: return "exact" when s.n == 10
    return "small" if s.n < 10 else "big"


g3.add_conditional_edges("check", router, {"small": "small", "big": "big"})  # TODO 3: add "exact"
g3.add_edge("small", END)
g3.add_edge("big", END)
# TODO 4: g3.add_edge("exact", END)
gate3 = g3.compile()
print("n=10 →", gate3.invoke(GateState(n=10))["verdict"], "  (goal: 'exact')")
# assert gate3.invoke(GateState(n=10))["verdict"] == "exact"   # un-comment when done

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
g3 = StateGraph(GateState)
for name, fn in [("check", check), ("small", small), ("big", big), ("exact", exact)]:
    g3.add_node(name, fn)
g3.set_entry_point("check")

def router(s):
    if s.n == 10:
        return "exact"
    return "small" if s.n < 10 else "big"

g3.add_conditional_edges("check", router, {"small": "small", "big": "big", "exact": "exact"})
for leaf in ("small", "big", "exact"):
    g3.add_edge(leaf, END)
gate3 = g3.compile()
assert gate3.invoke(GateState(n=10))["verdict"] == "exact"
```

Router label, mapping entry, node, exit edge — all four must agree, which is why the
consumer graph keeps its router a one-line lambda right next to the mapping.
</details>

## 6. The consumer graph, node by node

Now the real thing: `agents/src/agents/consumer_graph.py`. The docs name the flow
*discover → quote → decide → settle → activate → report*; the graph itself starts at
`request_quote` — **discovery** (finding Bell's card at its well-known URL) happens
*outside* the graph, over A2A (§9). Six nodes, one branch, one LLM slot:

- `request_quote` → `decide` → **accept?** → `settle` → `activate` → `report` → END
- **decline?** → `declined` → END — a graceful exit *before any money moves*

The outside world arrives behind a `Protocol` you can now read fluently (03):
`ConsumerTools` with `quote`/`settle`/`activate`. The graph never learns whether those
are stubs, or chainmcp + controller adapters (§8) — the same seam the controller's
ports gave you, paying off a third time. And the state is a dataclass where each node
fills the next field (`field(default_factory=list)` — 01's mutable-default fix):

In [ ]:
from agents.consumer_graph import ConsumerState, ConsumerTools, build_consumer_graph

print(inspect.getsource(ConsumerTools))
print(inspect.getsource(ConsumerState))

The factory builds the six node functions *inside* itself so they can capture `llm`
and `tools` — a **closure**, the trick you met building decorators in 01: a function
that remembers the variables of the scope it was born in. Read the whole factory and
find (a) the one node that calls `decide()` — the only LLM contact in the file — and
(b) the one-line lambda router that turns a decline into a graceful exit:

In [ ]:
print(inspect.getsource(build_consumer_graph))

To run it with no model and no chain we reuse the exact stand-ins from
`agents/tests/test_consumer_graph.py`: `_FakeLLM` (a scripted `structured()`, §4's
duck-typing again) and `_StubTools` (canned quote, `settle → 7` — ticket #7! —
`activate → "ent7-a0"`, plus two flags so we can *prove* later whether money moved).
Then let LangGraph draw the compiled topology as a Mermaid diagram — dotted arrows are
the conditional branch:

In [ ]:
class _FakeLLM:
    """An LLMClient stand-in: its structured() returns a fixed decision."""

    def __init__(self, accept):
        self._decision = DecisionOutput(accept=accept, reason="scripted for the test")

    def structured(self, system, user, schema):
        return self._decision


class _StubTools:
    def __init__(self):
        self.settled = False
        self.activated = False

    def quote(self, need):
        return CANONICAL_SIGNED_OFFER

    def settle(self, offer):
        self.settled = True
        return 7

    def activate(self, entitlement_id):
        self.activated = True
        return f"ent{entitlement_id}-a0"


tools_yes = _StubTools()
consumer_yes = build_consumer_graph(_FakeLLM(accept=True), tools_yes)
mermaid = consumer_yes.get_graph().draw_mermaid()
print(mermaid)
assert "decide -.->" in mermaid          # the dotted conditional edges out of `decide`

**The accept branch.** Invoke with the canonical need and a 15-TOK budget. The
transcript should read like the story's lifecycle in five lines, and the result dict
(§5's gotcha: a *dict*) should carry ticket #7 and session `ent7-a0` — the same
asserts as `test_happy_path_buys_and_activates`:

In [ ]:
result = consumer_yes.invoke(ConsumerState(need=BANDWIDTH_NEED, budget_tok=15))
for line in result["transcript"]:
    print("  " + line)

steps = [line.split(":")[0] for line in result["transcript"]]
assert steps == ["quote", "decide", "settle", "activate", "report"]
assert result["entitlement_id"] == 7 and result["session_id"] == "ent7-a0"
assert tools_yes.settled and tools_yes.activated
print("\n✓ five-line lifecycle — ticket #7 minted, session live, both tools touched")

**The decline branch.** Same graph shape, scripted `accept=False`. Watch what the
transcript *doesn't* contain — no settle, no activate — and check the stub's flags:
the decline exited **before any money moved**:

In [ ]:
tools_no = _StubTools()
consumer_no = build_consumer_graph(_FakeLLM(accept=False), tools_no)
result_no = consumer_no.invoke(ConsumerState(need=BANDWIDTH_NEED, budget_tok=15))
for line in result_no["transcript"]:
    print("  " + line)

assert [line.split(":")[0] for line in result_no["transcript"]] == ["quote", "decide", "exit"]
assert not tools_no.settled and not tools_no.activated
assert result_no["entitlement_id"] is None and result_no["session_id"] is None
print("\n✓ declined gracefully — settle/activate never ran, nothing purchased")

One asymmetry to notice, then break: **judgment failures decline safely (§4), but
infrastructure failures stay loud.** If the settlement tool blows up, the graph does
*not* catch it — a swallowed chain error would mean an agent that thinks it bought
something it didn't. Sabotage `settle` and watch the exception come straight through:

In [ ]:
class _ExplodingTools(_StubTools):
    def settle(self, offer):
        raise RuntimeError("chain adapter down")


exploding = build_consumer_graph(_FakeLLM(accept=True), _ExplodingTools())
try:
    exploding.invoke(ConsumerState(need=BANDWIDTH_NEED, budget_tok=15))
    raise AssertionError("should have raised")
except RuntimeError as e:
    print("✗ RuntimeError:", e, " ← the graph does NOT swallow it")
print("✓ decline is a branch; a broken tool is a crash — different failures, different manners")

**✏️ Your turn 6.1 — forget the budget, read the error**

`ConsumerState` has two required fields; every later field has a default. Delete
`budget_tok=15` inside the `try` block and re-run. Success: a `TypeError` whose
message *names the missing argument* — dataclasses tell you exactly which field you
owe them.

In [ ]:
state_ok = ConsumerState(need=BANDWIDTH_NEED, budget_tok=15)
print("unfilled fields start as None →", state_ok.offer, state_ok.decision, state_ok.entitlement_id)

message = ""
try:
    broken = ConsumerState(need=BANDWIDTH_NEED, budget_tok=15)   # TODO: delete `, budget_tok=15`
    print("constructed fine — now remove the budget and re-run")
except TypeError as e:
    message = str(e)
    print("✗ TypeError:", message)
# assert "budget_tok" in message   # un-comment when done

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
message = ""
try:
    broken = ConsumerState(need=BANDWIDTH_NEED)
except TypeError as e:
    message = str(e)
    print("✗ TypeError:", message)
assert "budget_tok" in message
```

`__init__() missing 1 required positional argument: 'budget_tok'` — required
dataclass fields are enforced at construction, so a graph can never start without a
budget for `decide()` to weigh.
</details>

## 7. The provider graph — two kinds of "no"

Flip to Bell's side. What breaks without this section: Bell's link carries 100 Mbps;
if his agent signs two 60-Mbps offers for the same window, the second buyer *paid
on-chain for capacity that does not exist*. "Please don't oversell" must not live in a
prompt — it must be **impossible by arithmetic**. So the provider graph has two gates,
different in kind:

1. **`admit` — deterministic.** A `CapacityLedger` does per-window bookkeeping; over
   capacity → immediate `Decline`, *no LLM asked*. Not judgment: arithmetic.
2. **`quote` — the second (and last) LLM slot.** Capacity confirmed, *price it* — or
   decline for business reasons.

The ledger first. Look for the dict **keyed by a `(start, end)` tuple** (reservations
only collide within the same window), `dict.get(key, 0)` as a zero-default
accumulator, and `try_reserve`'s all-or-nothing shape — check *then* write, 05's
atomicity discipline in miniature:

In [ ]:
from agents.provider_graph import (
    CapacityLedger,
    ProviderState,
    ProviderTools,
    QuoteDecision,
    build_provider_graph,
)

print(inspect.getsource(CapacityLedger))

Poke it with the canonical window: reserve 60 of 100, fail to reserve another 60 —
and verify the *failed* reserve left the ledger untouched — then fill the remaining
40 exactly:

In [ ]:
from a2a_interfaces.fixtures import WINDOW

W = (WINDOW.start, WINDOW.end)                     # reservations key on the exact window tuple
ledger = CapacityLedger(capacity_bps=100_000_000)
print("fresh ledger →", ledger.available(W) // 1_000_000, "Mbps free")
print("reserve 60   →", ledger.try_reserve(W, 60_000_000), "| now", ledger.available(W) // 1_000_000, "Mbps free")
print("reserve 60   →", ledger.try_reserve(W, 60_000_000), "| now", ledger.available(W) // 1_000_000, "Mbps free  ← refused, untouched")
print("_reserved    →", ledger._reserved, " ← the raw tuple-keyed dict (peeking)")
assert ledger.available(W) == 40_000_000
ok40 = ledger.try_reserve(W, 40_000_000)
print("reserve 40   →", ok40, "| now", ledger.available(W) // 1_000_000, "Mbps free  ← exactly full is allowed")
assert ok40 and ledger.available(W) == 0

Three small shapes around the graph, worth a quick read: `QuoteDecision` is the
provider LLM's output — a pydantic model kept **inside `agents`, never on the wire**
(what crosses to Ada is a `SignedOffer` or a `Decline`, §9); `ProviderTools` is a
one-method Protocol (sign an offer — the key stays in chainmcp, rule 2); and
`ProviderState` mirrors the consumer's dataclass with a three-way result field:

In [ ]:
print(inspect.getsource(QuoteDecision))
print(inspect.getsource(ProviderTools))
print(inspect.getsource(ProviderState))

Now the factory: `admit → quote → END`. Three details to find in the source:
`getattr(need, "capacity_bps", 0)` (duck-typed access — a `TelemetryNeed` has no such
field); the conditional edge where **`END` itself is a mapping key** (an inadmissible
request exits directly); and, in the `quote` node, `ledger.release(...)` when the LLM
declines — a business "no" must **give the tentatively-reserved slot back**:

In [ ]:
print(inspect.getsource(build_provider_graph))

Stubs, exactly as `agents/tests/test_provider_graph.py` builds them — `_FakeLLM` now
scripted with a `QuoteDecision` (we add one `calls` counter to the test's fake, so we
can *count* LLM consultations), `_SignTool` returning the canonical signed offer, and
the tests' `bandwidth_need_for` helper rebuilt by hand (it lives in
`agents/tests/tests_support.py`, importable only under pytest):

In [ ]:
from a2a_interfaces import BandwidthNeed, Decline, SignedOffer


class _FakeLLM:                                    # the provider tests' fake, plus a call counter
    def __init__(self, decision):
        self._decision = decision
        self.calls = 0

    def structured(self, system, user, schema):
        self.calls += 1
        return self._decision


class _SignTool:
    def sign_offer(self, need, price_tok):
        return CANONICAL_SIGNED_OFFER


def bandwidth_need_for(capacity_bps):
    """A need for a given rate in the canonical window — tests_support.py's helper, rebuilt."""
    return BandwidthNeed(src="hostA", dst="hostB", capacity_bps=capacity_bps, qos_class=1, window=WINDOW)


quote_llm = _FakeLLM(QuoteDecision(quote=True, price_tok=10, reason="scripted"))
bell_ledger = CapacityLedger(capacity_bps=100_000_000)
provider = build_provider_graph(quote_llm, _SignTool(), bell_ledger)
print("provider graph compiled — Bell's link: 100 Mbps | LLM consulted", quote_llm.calls, "times so far")

**The headline: no overselling, as code.** Ask for 60 Mbps twice against the 100-Mbps
ledger. The compiled graph *closes over one ledger*, so reservations persist between
invokes — that is the point. Watch three proofs land: the second result is a
`Decline`, its one-line transcript shows `admit` never reached `quote`, and the LLM
call counter **stays at 1** — the second "no" was arithmetic, not judgment:

In [ ]:
first = provider.invoke(ProviderState(need=bandwidth_need_for(60_000_000)))
print("request 1: 60 Mbps →", type(first["result"]).__name__,
      f"| {bell_ledger.available(W) // 1_000_000} Mbps free | {first['transcript']}")

second = provider.invoke(ProviderState(need=bandwidth_need_for(60_000_000)))
print("request 2: 60 Mbps →", type(second["result"]).__name__,
      f"| {bell_ledger.available(W) // 1_000_000} Mbps free | {second['transcript']}")

assert isinstance(first["result"], SignedOffer)
assert isinstance(second["result"], Decline) and "capacity" in second["result"].reason
assert "no overselling" in " ".join(second["transcript"])
assert quote_llm.calls == 1
print(f"\n✓ LLM consulted exactly {quote_llm.calls} time — the second refusal never woke it")

**The business decline gives the slot back.** Script the quote slot to say
`quote=False` ("rate too low this week") and watch the choreography: `admit` reserves
60 tentatively, `quote` declines, `release` returns the slot — available capacity ends
back at the full 100. Without that release, every business decline would silently
leak capacity:

In [ ]:
declining_llm = _FakeLLM(QuoteDecision(quote=False, price_tok=0, reason="rate too low this week"))
ledger2 = CapacityLedger(capacity_bps=100_000_000)
provider2 = build_provider_graph(declining_llm, _SignTool(), ledger2)

r = provider2.invoke(ProviderState(need=bandwidth_need_for(60_000_000)))
for line in r["transcript"]:
    print("  " + line)
print("result →", type(r["result"]).__name__, "|", r["result"].reason)
print("free   →", ledger2.available(W) // 1_000_000, "Mbps  ← the tentative reservation came back")
assert isinstance(r["result"], Decline) and ledger2.available(W) == 100_000_000

One last poke at the ledger's edge: `release` clamps at zero reserved
(`max(0, ...)`) — releasing more than was ever reserved cannot push availability past
capacity. The same defensive flavor as rule 8's idempotent teardown:

In [ ]:
ledger3 = CapacityLedger(capacity_bps=100_000_000)
ledger3.release(W, 999_999_999)                    # releasing what was never reserved
print("available →", ledger3.available(W) // 1_000_000, "Mbps  ← clamped, never above capacity")
assert ledger3.available(W) == 100_000_000

**✏️ Your turn 7.1 — predict the gate**

A fresh graph reserves 60 of 100. Write your predictions as a comment: does a 50-Mbps
request get an offer or a decline? What about 40 Mbps *after* the 50 was answered?
Then invoke and check. Success: your two predictions match the two printed result
types — and you can say why in one sentence about the ledger.

In [ ]:
led = CapacityLedger(capacity_bps=100_000_000)
gq = build_provider_graph(_FakeLLM(QuoteDecision(quote=True, price_tok=10, reason="scripted")),
                          _SignTool(), led)
opening = gq.invoke(ProviderState(need=bandwidth_need_for(60_000_000)))
print("60 Mbps →", type(opening["result"]).__name__, "| free:", led.available(W) // 1_000_000, "Mbps")
# my prediction: 50 Mbps → __________ , then 40 Mbps → __________
# TODO: fifty = gq.invoke(ProviderState(need=bandwidth_need_for(50_000_000))); print its result type
# TODO: forty = gq.invoke(ProviderState(need=bandwidth_need_for(40_000_000))); print its result type
# assert isinstance(fifty["result"], Decline) and isinstance(forty["result"], SignedOffer)  # un-comment

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
fifty = gq.invoke(ProviderState(need=bandwidth_need_for(50_000_000)))
print("50 Mbps →", type(fifty["result"]).__name__)
forty = gq.invoke(ProviderState(need=bandwidth_need_for(40_000_000)))
print("40 Mbps →", type(forty["result"]).__name__)
assert isinstance(fifty["result"], Decline) and isinstance(forty["result"], SignedOffer)
```

With 60 reserved, only 40 remain: 50 > 40 → declined (and, all-or-nothing, reserves
nothing), then 40 ≤ 40 → offered, filling the window exactly.
</details>

## 8. `mcp_tools.py` — the graphs get real hands (MCP, glossed)

So far every tool was a stub. **MCP — the Model Context Protocol** — is the standard
that makes the swap to real ones orderly: a way to expose *tools* (named operations
with JSON-shaped arguments) to agents, the way HTTP exposes pages to browsers. In 07
you drove `chainmcp`'s `ChainClient` by hand; `chain_tools(client)` wraps its five
operations as a dict of plain callables (and a FastMCP shell can serve the same dict
to an agent in another process).

The custody consequence is the point (rule 2): **each agent runs its own chainmcp
instance holding its own key.** Ada's tools close over Ada's client, Bell's over
Bell's — a graph only ever holds callables, and no key ever crosses the seam. Building
the dict touches nothing — the closures only reach for the client when *called* — so
we can inspect the tool surface with a placeholder object standing where a real
`ChainClient` would:

In [ ]:
from chainmcp.mcp_server import chain_tools

tool_surface = chain_tools(object())        # closures form now, touch the client only when called
print("the five operations →", sorted(tool_surface))
assert set(tool_surface) == {"sign_offer", "fulfill_offer", "read_entitlement",
                             "sign_activation_proof", "faucet"}
print("✓ a dict of callables closed over ONE agent's client — rule 2, in shape")

`ChainConsumerTools` is Ada's adapter: the same `ConsumerTools` shape the graph
already requires (§6), but real. Read it for three things: `settle()` is **one line**
— the graph node's `tools.settle(offer)` becomes `self._chain["fulfill_offer"](...)`,
a callable pulled out of the dict by string key; `activate()` is the deliberate
three-call dance from 08, now from the client side (challenge → sign the proof via
*chainmcp* → submit); and the injectable `http` client (same code drives a live
controller or an in-process one):

In [ ]:
from agents.mcp_tools import ChainConsumerTools, ChainProviderTools

print(inspect.getsource(ChainConsumerTools))

Stub and adapter, side by side. `ConsumerTools` is not `@runtime_checkable`, so no
`isinstance` here — check the structural fit by hand: both §6's `_StubTools` and this
chain-backed adapter carry the three required methods. That fit is the entire swap:
`build_consumer_graph(llm, ChainConsumerTools(...))` changes **zero** graph code:

In [ ]:
needed = [m for m in dir(ConsumerTools) if not m.startswith("_")]
print("ConsumerTools requires →", needed)
for impl in (_StubTools, ChainConsumerTools):
    fit = all(hasattr(impl, m) for m in needed)
    print(f"  {impl.__name__:<19} fits: {fit}")
    assert fit
print("✓ rule 7's spirit for agents: the graph cannot tell stub from chain — same seam, realer hands")

And one method refuses its role **on purpose**: `quote()` raises. The consumer must
never quote to itself — quotes arrive from the *other* agent, over A2A (§9). An
`NotImplementedError` with a reason is an architectural seam marker you can trip
today, no chain required (the constructor's placeholder client is never touched):

In [ ]:
consumer_tools = ChainConsumerTools(object(), controller_url="http://unused")
try:
    consumer_tools.quote(bandwidth_need_for(50_000_000))
    raise AssertionError("should have refused")
except NotImplementedError as e:
    print("✗ NotImplementedError:", e)
print("✓ an adapter refusing a role — the seam tells you where quotes really come from")

Bell's side: `ChainProviderTools.sign_offer` assembles a full canonical `Offer` — find
04's params packing reborn (`f"{CAPACITY_50_MBPS:064x}"`, two 32-byte words), the
zero-address open offer (`"0x" + "0" * 40`), and whole-TOK price times `10**18` into a
wei string — then hands it to chainmcp's `sign_offer` tool. The *assembly* is the
adapter's job; the *signing* is chainmcp's (07):

In [ ]:
print(inspect.getsource(ChainProviderTools))

Run the assembly line without a chain: a duck-typed stand-in provides the only two
members the adapter touches (`address`, `sign_offer`) and "signs" with 05's
placeholder signature. Everything *around* the signature is the adapter's real work,
and we can pin it against the fixtures:

In [ ]:
import a2a_interfaces.fixtures as fx
from a2a_interfaces import Offer


class _DuckChain:
    """Stands where Bell's chainmcp ChainClient would — the two members the adapter touches."""

    address = fx.BELL

    def sign_offer(self, offer):
        return SignedOffer(offer=offer, signature="0x" + "ab" * 65, terms_doc=fx.TERMS_DOC)


bell_tools = ChainProviderTools(_DuckChain())
signed = bell_tools.sign_offer(bandwidth_need_for(fx.CAPACITY_50_MBPS), price_tok=10)
print("provider →", signed.offer.provider[:10] + "…  == Bell?", signed.offer.provider == fx.BELL)
print("price    →", signed.offer.price, " ← 10 TOK as a wei string (04)")
print("params   →", signed.offer.params[:24] + "…  == canonical blob?",
      signed.offer.params == fx.BANDWIDTH_PARAMS_ABI)
assert signed.offer.price == fx.PRICE_10_TOK and signed.offer.params == fx.BANDWIDTH_PARAMS_ABI
print("✓ the adapter assembles; chainmcp signs — today's duck signs with 05's placeholder")

**✏️ Your turn 8.1 — pack a params blob yourself**

Rebuild 04's dissection in reverse: pack `(60_000_000 bps, qos_class 2)` into the
two-word hex blob the adapter would ship, then slice both numbers back out. Success:
the un-commented asserts pass — you can round-trip a params blob by hand.

In [ ]:
blob50 = "0x" + f"{fx.CAPACITY_50_MBPS:064x}" + f"{fx.QOS_CLASS:064x}"
print("canonical params rebuilt? →", blob50 == fx.BANDWIDTH_PARAMS_ABI)
# TODO: build blob60 the same way for 60 Mbps (60_000_000) and qos_class 2
# TODO: slice the numbers back out: int(blob60[2:66], 16) and int(blob60[66:130], 16)
# assert int(blob60[2:66], 16) == 60_000_000 and int(blob60[66:130], 16) == 2   # un-comment

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
blob60 = "0x" + f"{60_000_000:064x}" + f"{2:064x}"
print(blob60)
print("capacity →", int(blob60[2:66], 16), "| qos →", int(blob60[66:130], 16))
assert int(blob60[2:66], 16) == 60_000_000 and int(blob60[66:130], 16) == 2
```

Each `:064x` renders one right-aligned 32-byte ABI word (64 hex chars) — the same
two-word layout the contract decodes on-chain.
</details>

## 9. `a2a_adapter.py` — the envelope, never the letter (A2A, glossed)

The last missing piece of plumbing: how does Ada's agent *find* Bell's, and how does a
quote cross between two processes? **A2A** is an open agent-to-agent protocol (Linux
Foundation): every provider serves an **agent card** — a capability business card at a
well-known URL — listing its skills; a consumer fetches cards to *discover* who can
quote what, then exchanges messages whose data parts carry the payloads.

**ADR-002** disciplines the dependency: use the official SDK (`a2a-sdk`, pinned to
`0.3.26` — a young spec moves), but confine every SDK import to **one file**,
`a2a_adapter.py`. The SDK is the *envelope*; the letter is always our own pydantic
payloads (02), packed and unpacked at this single seam. Build Bell's card — pure
function, no server needed — and read it as docs/03 §1.1 would serve it:

In [ ]:
from agents.a2a_adapter import provider_card

card = provider_card("bandwidth-provider", "http://localhost:9101/", "bandwidth")
print("skill     →", card.skills[0].id, "|", card.skills[0].name)
print("streaming →", card.capabilities.streaming, " ← quotes are one-shot, no token streams")
print("modes     →", card.default_input_modes, card.default_output_modes)
assert card.skills[0].id == "quote_bandwidth" and card.capabilities.streaming is False

print("\nas the well-known endpoint would serve it:")
print(json.dumps(card.model_dump(exclude_none=True), indent=2))

"Imports confined to one file" is not a wish — it is *executable*. The CI test
`agents/tests/test_a2a.py` walks every module's **AST** (the parsed source tree — 08
used the same trick to prove the controller's purity) and fails if any file other than
the adapter imports the SDK. Run the same walk here:

In [ ]:
import ast
from pathlib import Path

import agents as agents_pkg

agents_src = Path(agents_pkg.__file__).parent
offenders = []
for py in sorted(agents_src.glob("*.py")):
    imports_sdk = any(
        (isinstance(node, ast.Import) and any(a.name.split(".")[0] == "a2a" for a in node.names))
        or (isinstance(node, ast.ImportFrom) and (node.module or "").split(".")[0] == "a2a")
        for node in ast.walk(ast.parse(py.read_text()))
    )
    print(("✗ imports the a2a SDK —" if imports_sdk else "✓ SDK-free            "), py.name)
    if imports_sdk and py.name != "a2a_adapter.py":
        offenders.append(py.name)
assert not offenders
print("\n✓ ADR-002 holds — the exact walk agents/tests/test_a2a.py runs in CI")

The letter itself: `encode_need` / `decode_need` and `encode_offer_or_decline` /
`decode_offer_or_decline` — the four pack/unpack functions. Two dispatch tricks worth
noticing in their source (both use *classes as first-class values*, 01): needs
dispatch on the `"kind"` discriminator you met in 02's tagged unions; offers-or-declines
dispatch on the presence of the `"declined": true` tag. Print the exact bytes that
cross between agents:

In [ ]:
from agents.a2a_adapter import (
    decode_need,
    decode_offer_or_decline,
    encode_need,
    encode_offer_or_decline,
)

wire = encode_need(BANDWIDTH_NEED)
print("a need on the wire:\n ", wire)
assert decode_need(wire) == BANDWIDTH_NEED

decline_wire = encode_offer_or_decline(Decline(reason="no capacity"))
print("\na decline on the wire:\n ", decline_wire, " ← 'declined': true is the dispatch tag")
assert decode_offer_or_decline(decline_wire) == Decline(reason="no capacity")
assert decode_offer_or_decline(encode_offer_or_decline(CANONICAL_SIGNED_OFFER)) == CANONICAL_SIGNED_OFFER
print("\n✓ lossless round trips for both payload kinds — the wire is just JSON strings")

Now the security story, end-to-end. Play the man-in-the-middle: intercept the signed
offer in transit and rewrite the price from 10 TOK to 1 TOK. Watch what the wire layer
does about it — **nothing**. It decodes fine. That is by design: the `SignedOffer`
carries its own EIP-712 signature over the *original* terms (07), so this layer needs
no trust of its own. The tampered offer walks free right up to the moment Ada tries to
redeem it — where the contract recovers a signer that isn't Bell and reverts with
`BadSignature`. You built that trap in 07; `test_a2a.py` springs it on a live Anvil
(our fixture's signature is 05's placeholder, so the chain leg needs the real signing
from 07 first).

In [ ]:
on_the_wire = encode_offer_or_decline(CANONICAL_SIGNED_OFFER)
blob = json.loads(on_the_wire)
blob["offer"]["price"] = str(10**18)               # the MITM rewrites 10 TOK → 1 TOK
tampered = decode_offer_or_decline(json.dumps(blob))

print("decoded   →", type(tampered).__name__, " ← no error at all")
print("price     →", int(tampered.offer.price) // 10**18, "TOK  (was 10)")
print("signature → unchanged?", tampered.signature == CANONICAL_SIGNED_OFFER.signature,
      " ← still signs the ORIGINAL terms")
assert isinstance(tampered, SignedOffer) and tampered.offer.price == str(10**18)
assert tampered.signature == CANONICAL_SIGNED_OFFER.signature
print("\nthe wire is trustless on purpose — the CONTRACT is the checkpoint (BadSignature, 07)")

**✏️ Your turn 9.1 — remove the dispatch tag, read the fallout**

The `"declined"` key is load-bearing: without it, `decode_offer_or_decline` assumes
the payload must be a `SignedOffer` and validates it as one. Delete the key from an
encoded `Decline` and decode. Success: a `ValidationError` — and its error list tells
you which *model's* required fields went missing.

In [ ]:
payload = json.loads(encode_offer_or_decline(Decline(reason="over capacity")))
print("keys on the wire →", sorted(payload))
caught = None
# TODO: del payload["declined"], then decode_offer_or_decline(json.dumps(payload))
#       inside try/except ValidationError — store the exception in `caught`
# assert caught is not None and "offer" in str(caught)   # un-comment when done

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
del payload["declined"]
try:
    decode_offer_or_decline(json.dumps(payload))
except ValidationError as e:
    caught = e
    print("✗ ValidationError —", len(e.errors()), "missing fields, e.g.", e.errors()[0]["loc"])
assert caught is not None and "offer" in str(caught)
```

Without the tag, dispatch picks `SignedOffer` and the validator demands its fields
(`offer`, `signature`, …) — dispatch chooses the class first, validation judges second.
</details>

## 10. Where judgment lives — the whole map

Zoom out. Every decision the two graphs (and the organs beneath them) make, sorted
into rule 1's two piles:

| Step / decision | Who | Deterministic or LLM? | Where |
|---|---|---|---|
| discovery (fetch agent cards) | consumer | deterministic | `a2a_adapter.py` (§9) |
| `request_quote` (ask, receive) | consumer | deterministic | `consumer_graph.py` |
| **accept / reject the offer** | consumer | **LLM — slot 1** | `decision.py` (§4) |
| `settle` / `activate` / `report` / `declined` | consumer | deterministic | `consumer_graph.py` + tools (§8) |
| `admit` (no overselling) | provider | deterministic — ledger arithmetic | `provider_graph.py` (§7) |
| **quote / decline (pricing)** | provider | **LLM — slot 2** | `provider_graph.py` (§7) |
| offer validity, payment, mint | contract | deterministic | `Settlement.sol` (06/07) |
| authorize activation | controller | deterministic — the predicate | `controller/domain.py` (08) |
| configure the router | netctl | deterministic | `GnmiProvisioner` (09) |

Exactly **two** LLM cells, both of them *commercial* decisions — whether a deal
happens and at what price. Everything the network *does* is deterministic and
replayable. Judgment proposes; the predicate (and the contract) dispose.

**✏️ Your turn 10.1 — prove the count, don't trust the table**

Only `llm.py` defines `structured()`; judgment happens wherever some file *calls*
`.structured(`. Predict which files those are (write the comment first), then run the
scan. The active assert pins rule 1's headline; un-comment the second to pin your
exact prediction.

In [ ]:
# my prediction: the files that CALL .structured( are ________ and ________
call_sites = {}
for py in sorted(agents_src.glob("*.py")):
    hits = py.read_text().count(".structured(")
    if hits:
        call_sites[py.name] = hits
print("call sites →", call_sites)
assert sum(call_sites.values()) == 2               # rule 1: exactly two judgment slots
# assert call_sites == {"decision.py": 1, "provider_graph.py": 1}   # un-comment after predicting

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
assert call_sites == {"decision.py": 1, "provider_graph.py": 1}
```

The consumer's slot lives in its own module (`decision.py`); the provider's is the
`quote` node inside `provider_graph.py`. `consumer_graph.py` never calls
`.structured(` itself — it goes through `decide()`, keeping the slot in one place.
</details>

## 11. Pointing the same brains at a real model

Everything above ran on scripts. Arming real judgment changes **no code** — only the
environment (ADR-001's whole point). Two legs, from §2's table:

```bash
# Leg A — local Ollama (a free, small model on your own machine)
ollama pull qwen3:4b                                # once
export LLM_BASE_URL=http://localhost:11434/v1
export LLM_MODEL=qwen3:4b
export LLM_API_KEY=ollama                           # Ollama ignores it, the shape demands it

# Leg B — the deployed Modal vLLM endpoint (llmserve/modal_llm.py, the live-demo leg)
export LLM_BASE_URL=https://<workspace>--a2a-llm-serve.modal.run/v1
export LLM_MODEL=qwen3-4b                           # the served alias — SERVED_AS in llmserve/modal_llm.py
export LLM_API_KEY=<modal proxy token>

# then, either leg — arm the judgment slots and re-run this notebook's final cell:
export A2A_LIVE_LLM=1
```

(The `just up` deployment loads the same three vars from a repo-root `.env`, probes
the endpoint with `agents.llm.llm_up` — a 1-token completion with a deadline — and
treats a slow first answer as *warming*, not absent: Modal containers boot on demand,
ADR-001's amendment.)

Why the swap is safe: **the validator makes backend differences irrelevant.** Ollama
wraps answers in `<think>` blocks; vLLM honors schemas differently; a tired model
emits half-JSON — every flavor of wander dies at the same fence you exhausted in §3,
and the graphs only ever see a `DecisionOutput`/`QuoteDecision` or a clean safe
default. Same code, three interchangeable backends, one guarantee.

The cell below is **armed but asleep**: headless runs (and yours, unless you opt in)
skip it. With `A2A_LIVE_LLM=1` and a serving endpoint it runs §4's judgment twice —
the same 10-TOK offer against a 15-TOK and then a 5-TOK budget — and you watch the one
genuine judgment slot flip where the scripted spy of 4.1 could not:

In [ ]:
import os

from agents.llm import llm_up

if os.environ.get("A2A_LIVE_LLM") == "1" and llm_up():
    live = LLMClient()                      # LLMConfig.from_env() → whatever backend env names
    for budget in (15, 5):
        v = decide(live, BANDWIDTH_NEED, CANONICAL_SIGNED_OFFER, budget_tok=budget)
        print(f"budget {budget:>2} TOK on the 10-TOK offer → accept={v.accept}  ({v.reason})")
else:
    print("skipped: needs A2A_LIVE_LLM=1 and a serving endpoint —")
    print("         set the three LLM_* vars (recipe above), export A2A_LIVE_LLM=1, re-run this cell")

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| built a chat request and response by hand | an LLM is a next-word predictor; the API is a list of role+content dicts | every judgment call in the system |
| resolved `LLMConfig.from_env()` | ADR-001: one dialect, backend chosen by three env vars | `just up` wiring (11), the live demo (12) |
| ran the scripted validate-and-retry loop to attempt 3 | text in, **types** out — pydantic is the fence (02's payoff) | both judgment slots, and 13's LLM-accuracy experiment |
| exhausted the budget → `StructuredError.attempts` | fail with evidence, then a safe default | `decide()`'s decline fallback |
| printed `decide()`'s exact prompt | slot 1: the consumer's accept/reject; judgment proposes, the predicate disposes | armed live in 11/12 |
| built toy graphs, then both consumer branches | nodes are functions, state flows, a dict comes back; decline exits before money | the full profile's lifecycle (11/12) |
| 60+60 against 100 → second declined, LLM asleep | admission control is arithmetic, not judgment — no overselling | the story's ch. 8 promise, measured in 13 |
| duck-typed a chain into `ChainProviderTools` | MCP tools: same Protocol seam, keys stay in chainmcp (rule 2) | the `chain` worlds (11) |
| walked every module with `ast` | ADR-002: the A2A SDK lives in exactly one file | the discovery wiring of the finale (12) |
| tampered a wire offer — it decoded fine | the wire is trustless; the contract verifies (`BadSignature`) | the adversarial matrix (13/14) |
| counted `.structured(` call sites: two | **rule 1, proven** — exactly two LLM cells | the "where judgment lives" claim in the evaluation |

## Experiments to try

Predict the outcome *before* running each one:

- Make `_StubTools.activate` raise `RuntimeError` and run the accept branch. Does the
  transcript show `settle` before the crash — i.e. did money move before activation
  died? (This is why the real teardown/activation story of 08/12 matters.)
- In 6's graph, replace the conditional-edge lambda with `lambda s: "settle"` and run
  the *decline* transcript. Watch a "declined" decision buy anyway — one line was the
  entire fence between judgment and money.
- Rebuild §3's client with `max_retries=1` and the three-reply script. Which exception,
  and how many entries in `.attempts`?
- Deliberate breakage: in §7, script the LLM with `QuoteDecision(quote=False, ...)` but
  call `build_provider_graph` with a ledger you *pre-fill* via `try_reserve(W, 100_000_000)`.
  Which gate answers, and does the transcript ever mention the quote?

## Check yourself

1. In one honest sentence: what is an LLM, and why does this system still work if it
   hallucinates?
2. The model replies `{"accept": false}` — valid JSON. Why does `structured()` burn a
   retry anyway, and which layer refused it?
3. The provider says "no" twice in §7's headline run. Which "no" was judgment, which
   was arithmetic — and how did the call counter prove it?
4. `result = graph.invoke(ConsumerState(...))` — what *type* is `result`, and how do
   you read the transcript out of it?
5. A man-in-the-middle rewrites an offer's price in transit. Name the exact place the
   fraud dies, and why `a2a_adapter.py` deliberately does nothing about it.

**Where this goes next:** `11_worlds_and_profiles.ipynb` — the composition root:
`SKELETON_PROFILE` swaps 05's fakes, these graphs, and the real chain organs in and
out of one unchanged lifecycle.

**Deeper dive:** the compact tour
[`e2e/notebooks/agents_explore.ipynb`](../agents_explore.ipynb) · the spec
[`docs/06-agents-spec.md`](../../../docs/06-agents-spec.md) ·
[`docs/adr/001-llm-serving.md`](../../../docs/adr/001-llm-serving.md) and
[`docs/adr/002-a2a-sdk.md`](../../../docs/adr/002-a2a-sdk.md) · the stub patterns you
reused: [`agents/tests/`](../../../agents/tests/) · story ch. 8–9.